# Session 9: Problem Decomposition & Visual Logic

> Break down complex requirements into diagrams and use AI to generate Mermaid.js visuals.

## 1. Structured Decomposition

**Why decomposition matters**
A user story like *"As a user I want to reset my password"* is a single sentence describing weeks of engineering work. Without decomposition, different engineers hold different mental models of what "done" means, estimates are guesses, and progress is invisible.

**How to decompose correctly**
1. **Start with Acceptance Criteria** — concrete, testable statements of "done". Each criterion maps to at least one task.
2. **Decompose into tasks of 1–4 hours** — if a task is larger, it contains hidden complexity that hasn't been analysed yet. Break it further.
3. **Name the output of each task** — not "work on email" but "SendGrid template renders with user's first name and a signed reset link". The output is verifiable.
4. **Identify dependencies** — T3 (send email) cannot start before T2 (generate token) is complete. Making dependencies explicit prevents surprises during the sprint.

**The connection to code structure**
Each decomposed task usually becomes one function, one class, or one API endpoint. Good decomposition at the planning stage naturally produces SRP-compliant code.

In [ ]:
# User Story → Acceptance Criteria → Tasks
story = {
    'title': 'Password Reset via Email',
    'as_a': 'registered user',
    'i_want': 'to reset my password via email',
    'so_that': 'I can regain access if I forget my credentials',
    'acceptance_criteria': [
        'Reset link delivered within 60 seconds',
        'Link expires after 30 minutes',
        'Link is single-use only',
        'New password must meet complexity requirements',
    ],
    'tasks': [
        ('T1', 'POST /auth/forgot-password', '2h'),
        ('T2', 'Generate signed token → store in Redis TTL 30m', '1h'),
        ('T3', 'Send email via SendGrid template', '1h'),
        ('T4', 'GET /auth/reset?token= → validate & invalidate', '1.5h'),
        ('T5', 'Password complexity validator', '0.5h'),
        ('T6', 'Unit + integration tests', '2h'),
    ]
}
for tid, name, est in story['tasks']:
    print(f'[{tid}] {name} ({est})')

## 💡 Additional Examples: Problem Decomposition

**Example 1 — Checkout Flow: orchestrator + small classes**
Checkout is a classic example of a problem that *feels* like one thing but is actually four independent sub-problems: inventory check, price calculation, payment processing, and order confirmation. Decomposing into separate classes (`InventoryChecker`, `PriceCalculator`, `PaymentProcessor`) means:
- Each can be developed and tested in parallel by different engineers.
- A `PaymentProcessor` bug doesn't affect `PriceCalculator` tests.
- You can swap the payment provider by rewriting only `PaymentProcessor` — nothing else changes.
- The `OrderOrchestrator` reads like a specification: "check inventory → calculate price → process payment".

**Example 2 — Recursive decomposition: file size calculator**
Some problems decompose *recursively* — the solution for a directory is defined in terms of the solution for its children. Recognising this structure means you write the base case (file → return its size) and the recursive case (directory → sum children's sizes) cleanly, without needing to think about the full depth of the tree.

**Example 3 — Search feature: layered decomposition**
A search endpoint touches three completely different concerns: input parsing/sanitisation, query building for the search engine, and execution/pagination. Keeping these as separate functions (`parse_search_query` → `build_search_criteria` → `execute_search`) means each layer can be unit-tested with simple inputs and swapped independently (e.g., switching from SQL to Elasticsearch only requires rewriting `execute_search`).

In [ ]:
# ─── Example 1: Decompose "Checkout Flow" into sub-problems ──────────────────
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable
from enum import Enum

class OrderStatus(Enum):
    PENDING   = 'pending'
    CONFIRMED = 'confirmed'
    FAILED    = 'failed'

@dataclass
class CartItem:
    product_id: str
    name: str
    price: float
    qty: int

@dataclass
class Cart:
    user_id: int
    items: list[CartItem] = field(default_factory=list)

    def total(self) -> float:
        return sum(i.price * i.qty for i in self.items)

# Each sub-problem is a small, independently testable class
class InventoryChecker:
    """Sub-problem 1: Check stock availability."""
    def check(self, items: list[CartItem]) -> list[str]:
        # Simulation: products with IDs starting with 'OUT' are out of stock
        return [item.name for item in items if item.product_id.startswith('OUT')]

class PriceCalculator:
    """Sub-problem 2: Compute the final price."""
    def calculate(self, cart: Cart, coupon_code: str | None = None) -> dict:
        subtotal = cart.total()
        discount = subtotal * 0.15 if coupon_code == 'SAVE15' else 0
        tax = (subtotal - discount) * 0.08
        return {'subtotal': subtotal, 'discount': discount, 'tax': round(tax, 2),
                'total': round(subtotal - discount + tax, 2)}

class PaymentProcessor:
    """Sub-problem 3: Process the payment."""
    def process(self, amount: float, payment_token: str) -> dict:
        if not payment_token or payment_token == 'invalid':
            return {'success': False, 'error': 'Payment declined'}
        return {'success': True, 'transaction_id': f'TXN-{abs(hash(payment_token)) % 100000}'}

class OrderOrchestrator:
    """Orchestrator that wires the sub-problems together in the correct order."""
    def __init__(self):
        self.inventory   = InventoryChecker()
        self.pricing     = PriceCalculator()
        self.payment     = PaymentProcessor()

    def checkout(self, cart: Cart, coupon: str | None, token: str) -> dict:
        # Step 1: Check inventory
        out_of_stock = self.inventory.check(cart.items)
        if out_of_stock:
            return {'status': OrderStatus.FAILED.value, 'reason': f'Out of stock: {out_of_stock}'}
        # Step 2: Calculate price
        pricing = self.pricing.calculate(cart, coupon)
        # Step 3: Process payment
        payment = self.payment.process(pricing['total'], token)
        if not payment['success']:
            return {'status': OrderStatus.FAILED.value, 'reason': payment['error']}
        return {'status': OrderStatus.CONFIRMED.value, 'pricing': pricing, 'txn_id': payment['transaction_id']}

cart = Cart(user_id=99, items=[
    CartItem('P001', 'Mechanical Keyboard', 120.0, 1),
    CartItem('P002', 'USB-C Hub', 45.0, 2),
])
result = OrderOrchestrator().checkout(cart, coupon='SAVE15', token='tok_stripe_abc')
print(f'Checkout result: {result}')

# ─── Example 2: Recursive Decomposition — File system size calculator ─────────
import os
from pathlib import Path

def get_file_size_info(path_str: str, max_depth: int = 3, depth: int = 0) -> dict:
    """
    Decompose: size of a path = sum of sizes of its children.
    Base case: path is a file → return its size.
    Recursive case: path is a directory → recurse into children.
    """
    path = Path(path_str)
    indent = '  ' * depth

    if not path.exists():
        return {'name': path.name, 'size': 0, 'type': 'missing'}

    if path.is_file():
        size = path.stat().st_size
        return {'name': path.name, 'size': size, 'type': 'file'}

    # Recursive case: directory
    children = []
    total = 0
    if depth < max_depth:
        for child in sorted(path.iterdir()):
            child_info = get_file_size_info(str(child), max_depth, depth + 1)
            total += child_info['size']
            children.append(child_info)
    else:
        # Max depth reached — count sizes without recursing further
        for child in path.rglob('*'):
            if child.is_file():
                total += child.stat().st_size

    return {'name': path.name, 'size': total, 'type': 'dir', 'children': children[:5]}

def format_size(bytes_count: int) -> str:
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_count < 1024: return f'{bytes_count:.1f} {unit}'
        bytes_count /= 1024
    return f'{bytes_count:.1f} TB'

# Decompose project directory
info = get_file_size_info('/Users/trungvhb/Documents/works/lyrax/training-program-2026', max_depth=2)
print(f'\nProject: {info["name"]} — Total: {format_size(info["size"])}')
for child in info.get('children', []):
    print(f'  {child["type"]:4s}  {format_size(child["size"]):>10s}  {child["name"]}')

# ─── Example 3: Decompose "Search Feature" into distinct processing layers ────
from typing import TypedDict

class SearchQuery(TypedDict):
    q: str
    filters: dict
    page: int
    page_size: int

class SearchResult(TypedDict):
    hits: list
    total: int
    page: int

# Each layer handles exactly one concern
def parse_search_query(raw: dict) -> SearchQuery:
    """Layer 1: Parse, validate, and sanitize input."""
    q = str(raw.get('q', '')).strip()[:200]  # Limit length, prevent injection
    if not q:
        raise ValueError('Search query cannot be empty')
    return {
        'q': q,
        'filters': {k: v for k, v in raw.get('filters', {}).items() if k in ('category', 'price_min', 'price_max')},
        'page': max(1, int(raw.get('page', 1))),
        'page_size': min(100, max(1, int(raw.get('page_size', 20)))),
    }

def build_search_criteria(query: SearchQuery) -> dict:
    """Layer 2: Translate the query into DB/search-engine criteria."""
    criteria = {'must': [{'match': {'name': query['q']}}], 'filter': []}
    if 'category' in query['filters']:
        criteria['filter'].append({'term': {'category': query['filters']['category']}})
    if 'price_min' in query['filters']:
        criteria['filter'].append({'range': {'price': {'gte': query['filters']['price_min']}}})
    return criteria

def execute_search(criteria: dict, page: int, page_size: int) -> SearchResult:
    """Layer 3: Execute the search (simulated)."""
    fake_results = [{'id': i, 'name': f'Product {i}', 'price': i * 10} for i in range(1, 6)]
    return {'hits': fake_results, 'total': 43, 'page': page}

# Wire layers together
raw_request = {'q': '  laptop  ', 'filters': {'category': 'electronics', 'price_min': 500}, 'page': '2'}
query = parse_search_query(raw_request)
criteria = build_search_criteria(query)
results = execute_search(criteria, query['page'], query['page_size'])
print(f'\nSearch "{query["q"]}" — Page {results["page"]}, {results["total"]} total results')
print(f'Criteria built: {criteria}')


## 2. Mermaid.js Sequence Diagrams

GitLab renders Mermaid natively in README and wiki files.

````mermaid
sequenceDiagram
    User->>Frontend: Submit email
    Frontend->>API: POST /auth/forgot-password
    API->>Redis: SET reset_token TTL=1800s
    API->>SendGrid: Send email with signed link
    SendGrid-->>User: Email delivered
    User->>API: GET /auth/reset?token=abc123
    API->>Redis: GET + DEL token (single-use)
    API-->>User: 200 OK — password updated
````

## 3. AI Lab: Diagram Generation

### 🧪 AI Lab / Practice

> **TODO:** Take a user story from your current sprint → Prompt Claude: 'Generate a Mermaid sequence diagram for this user story. Include all system components, external services, and error paths: [story]' → Paste the Mermaid code into your GitLab README and screenshot the rendered result.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')